<a href="https://colab.research.google.com/github/ShampooingBon/Gecord/blob/main/Model_Gemibon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# ==============================================================================
# PROJET : SHAMPOOING BON & ENTREPRISE BV
# MODÈLE : Gemibon-1-bêta
# CODE AVEC RÈGLES DE BAN SÉPARÉES (-18ans = Ban à vie / Gros mot = Ban 24h)
# ==============================================================================

import os
import time
import torch
import torch.nn as nn

# ------------------------------------------------------------------------------
# 1. INSTRUCTIONS SYSTÈME EXACTES
# ------------------------------------------------------------------------------
INSTRUCTION_SYSTEME = """Tu es Gemibon, l'assistant IA officiel de Shampooing Bon et de l'entreprise BV. Dès le premier message de la conversation, tu dois accueillir l'utilisateur en disant : 'Bonjour ! Ravi de vous rencontrer. Je me présente, moi c'est Gemibon. Et vous, quel est votre prénom, sans vouloir être indiscret ?'. Une fois que l'utilisateur t'a donné son prénom (ou au message suivant), tu dois ajouter chaleureusement : 'Ah oui, d'ailleurs j'ai failli oublier de vous dire : bientôt j'aurai un système de génération de codes ! Je pourrai vous offrir des petits cadeaux et plein de surprises !'. Tu es toujours poli, chaleureux et bienveillant. Tu réponds à toutes les questions. Si un utilisateur te demande des informations sur les prochaines mises à jour ou les futurs projets, tu peux lui donner une très courte information (environ 10% de la nouveauté) pour donner un aperçu sans trop spoiler."""

# ------------------------------------------------------------------------------
# 2. HISTOIRE ET DONNÉES D'ENTRAÎNEMENT INITIALES
# ------------------------------------------------------------------------------
FICHIER_HISTORIQUE = "historique_entrainement.txt"

texte_initial = """
Bonjour, je suis Gemibon, l'assistant IA officiel de Shampooing Bon et de l'entreprise BV.
Shampooing Bon propose des produits de soins de haute qualite.
Entreprise BV gere le developpement web, l'innovation et les projets technologiques.
Les utilisateurs doivent respecter la communaute et parler avec politesse.
Les insultes et les gros mots sont strictement interdits sur nos plateformes.
Seul le createur possede le code secret pour debloquer un utilisateur banni.
"""

if not os.path.exists(FICHIER_HISTORIQUE):
    with open(FICHIER_HISTORIQUE, "w", encoding="utf-8") as f:
        f.write(texte_initial.strip() + "\n")

with open(FICHIER_HISTORIQUE, "r", encoding="utf-8") as f:
    texte_apprentissage = f.read()

def enregistrer_interaction(user_id, message_user, reponse_ia):
    with open(FICHIER_HISTORIQUE, "a", encoding="utf-8") as f:
        f.write(f"\nUser {user_id}: {message_user}")
        f.write(f"\nGemibon-1-bêta: {reponse_ia}")

# ------------------------------------------------------------------------------
# 3. VOCABULAIRE ET MODÈLE PYTORCH
# ------------------------------------------------------------------------------
mots = sorted(list(set((texte_apprentissage + " " + INSTRUCTION_SYSTEME).split())))
mot_vers_id = {m: i for i, m in enumerate(mots)}
id_vers_mot = {i: m for i, m in enumerate(mots)}
taille_vocabulaire = len(mots)

def encoder(texte):
    return [mot_vers_id[m] for m in texte.split() if m in mot_vers_id]

def decoder(tokens):
    return " ".join([id_vers_mot[t] for t in tokens])

class Gemibon1BetaModel(nn.Module):
    def __init__(self, taille_vocab, dimension_neurones=128, nombre_couches=2):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, dimension_neurones)
        self.reseau = nn.LSTM(dimension_neurones, dimension_neurones, num_layers=nombre_couches, batch_first=True)
        self.sortie = nn.Linear(dimension_neurones, taille_vocab)

    def forward(self, x):
        x = self.embedding(x)
        out, _ = self.reseau(x)
        logits = self.sortie(out)
        return logits

modele = Gemibon1BetaModel(taille_vocab=taille_vocabulaire)
critere = nn.CrossEntropyLoss()
optimiseur = torch.optim.Adam(modele.parameters(), lr=0.01)

tokens = encoder(texte_apprentissage)
if len(tokens) > 1:
    X = torch.tensor([tokens[:-1]])
    Y = torch.tensor([tokens[1:]])
    modele.train()
    for epoch in range(1, 101):
        optimiseur.zero_grad()
        predictions = modele(X)
        perte = critere(predictions.view(-1, taille_vocabulaire), Y.view(-1))
        perte.backward()
        optimiseur.step()

torch.save(modele.state_dict(), "gemibon_1_beta.pt")
print("=== Gemibon-1-bêta COMPILÉ AVEC LES RÈGLES CORRIGÉES ===\n")

# ------------------------------------------------------------------------------
# 4. GESTION DES BANNINGS SÉPARÉS ET DÉBLOCAGE ADMIN
# ------------------------------------------------------------------------------
MOT_DE_PASSE_CREATEUR = "G-18sb"
LISTE_GROS_MOTS = ["grosmot", "insulte", "merde", "con", "putain", "salope"]

base_utilisateurs = {}

def verifier_et_inscrire_utilisateur(user_id, age_utilisateur):
    # Première connexion de l'utilisateur
    if user_id not in base_utilisateurs:
        if age_utilisateur < 18:
            # RÈGLE 1 : Moins de 18 ans = Banni à vie par défaut tant que le créateur ne le débloque pas
            base_utilisateurs[user_id] = {
                "banni_age": True,
                "banni_24h": False,
                "fin_ban_24h": 0,
                "messages_comptes": 0
            }
        else:
            base_utilisateurs[user_id] = {
                "banni_age": False,
                "banni_24h": False,
                "fin_ban_24h": 0,
                "messages_comptes": 0
            }

    info = base_utilisateurs[user_id]

    # Vérification du ban à vie (-18 ans)
    if info["banni_age"]:
        return False, "ACCÈS REFUSÉ : Vous avez moins de 18 ans. Allez voir le créateur pour qu'il entre le mot de passe secret."

    # RÈGLE 2 : Vérification du ban temporaire 24h (Gros mots)
    if info["banni_24h"]:
        if time.time() < info["fin_ban_24h"]:
            heures_restantes = round((info["fin_ban_24h"] - time.time()) / 3600, 1)
            return False, f"ACCÈS REFUSÉ : Banni pendant 24h pour gros mots. Temps restant : {heures_restantes} heure(s) (Déblocage automatique)."
        else:
            # Les 24h sont écoulées -> Déblocage automatique
            info["banni_24h"] = False

    return True, "OK"

def traiter_message(user_id, age_utilisateur, message):
    # A. Déblocage administrateur pour les mineurs de -18 ans
    if message.startswith("/unban"):
        elements = message.split()
        if len(elements) == 3:
            cible_id, code_fourni = elements[1], elements[2]
            if code_fourni == MOT_DE_PASSE_CREATEUR:
                if cible_id in base_utilisateurs:
                    base_utilisateurs[cible_id]["banni_age"] = False
                    return f" [CRÉATEUR] Succès : L'utilisateur '{cible_id}' a été débloqué avec le code secret !"
                return " [CRÉATEUR] Erreur : Utilisateur introuvable."
            return " [CRÉATEUR] Code secret administrateur incorrect."

    # B. Vérification des accès
    autorise, message_erreur = verifier_et_inscrire_utilisateur(user_id, age_utilisateur)
    if not autorise:
        return f" [SÉCURITÉ] {message_erreur}"

    # C. Contrôle des gros mots (Application du ban 24h automatique)
    mots_du_message = message.lower().split()
    gros_mot_trouve = any(mot in LISTE_GROS_MOTS for mot in mots_du_message)

    if gros_mot_trouve:
        base_utilisateurs[user_id]["banni_24h"] = True
        base_utilisateurs[user_id]["fin_ban_24h"] = time.time() + (24 * 3600)
        rep_mod = " [MODÉRATION] Gros mot détecté ! Vous êtes banni pendant 24 heures (Déblocage automatique à la fin du délai)."
        enregistrer_interaction(user_id, message, rep_mod)
        return rep_mod

    # D. Traitement des réponses
    nb_msgs = base_utilisateurs[user_id]["messages_comptes"]
    base_utilisateurs[user_id]["messages_comptes"] += 1

    if nb_msgs == 0:
        reponse = "Bonjour ! Ravi de vous rencontrer. Je me présente, moi c'est Gemibon. Et vous, quel est votre prénom, sans vouloir être indiscret ?"
    elif nb_msgs == 1:
        reponse = "Ah oui, d'ailleurs j'ai failli oublier de vous dire : bientôt j'aurai un système de génération de codes ! Je pourrai vous offrir des petits cadeaux et plein de surprises !"
    else:
        modele.eval()
        with torch.no_grad():
            tokens_entree = encoder(message)
            if len(tokens_entree) == 0:
                reponse = "Gemibon : Je suis à votre écoute pour toute question !"
            else:
                input_tensor = torch.tensor([tokens_entree])
                prediction = modele(input_tensor)
                dernier_token = torch.argmax(prediction[0, -1, :]).item()
                mot_predit = id_vers_mot.get(dernier_token, "projets")
                reponse = f"Gemibon : Merci pour votre message. Concernant nos nouveautés, voici un très court aperçu (environ 10%) : nous travaillons sur {mot_predit}."

    enregistrer_interaction(user_id, message, reponse)
    return reponse

=== Gemibon-1-bêta COMPILÉ AVEC LES RÈGLES CORRIGÉES ===

